## Notebook 04 — Silver to Gold (Dimensional Model)
### Part 3: Load dimensional model from Silver (cleansed) to Gold

**Star Schema (matching dimensional model diagram):**
- `dim_date` — calendar attributes
- `dim_inspection_type` — inspection type + city
- `dim_inspection_result` — result type + city
- `dim_location` — address, city, state, zip, lat/lon
- `dim_violation` — violation code/desc, both cities (no need to match codes)
- `dim_establishment` — **SCD Type 2** (DBA, AKA, license, facility, risk)
- `fact_inspection` — grain: one row per inspection
- `bridge_inspection_violation` — maps inspections ↔ violations (many-to-many)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

CATALOG = "chicago_dallas_food_inspection"

chi_insp = spark.table(f"{CATALOG}.silver.chicago_inspections")
dal_insp = spark.table(f"{CATALOG}.silver.dallas_inspections")
chi_viol = spark.table(f"{CATALOG}.silver.chicago_violations")
dal_viol = spark.table(f"{CATALOG}.silver.dallas_violations")

print(f"Chicago inspections: {chi_insp.count()}, violations: {chi_viol.count()}")
print(f"Dallas inspections:  {dal_insp.count()}, violations: {dal_viol.count()}")

### 4.1 — dim_date

In [0]:
chi_dt = chi_insp.select(col("Inspection_Date").alias("dt"))
dal_dt = dal_insp.select(col("Inspection_Date").alias("dt"))
all_dt = chi_dt.union(dal_dt).distinct().filter(col("dt").isNotNull())

dim_date = (all_dt
    .withColumn("DATE_SK", (year("dt")*10000 + month("dt")*100 + dayofmonth("dt")).cast("int"))
    .withColumn("DI_CREATEDATE", current_timestamp())
    .withColumn("FULL_DT", col("dt"))
    .withColumn("DAY_NUM", dayofmonth("dt"))
    .withColumn("DAY_NAME", date_format("dt", "EEEE"))
    .withColumn("IS_WEEKEND", when(dayofweek("dt").isin(1,7), "Y").otherwise("N"))
    .withColumn("WEEK_OF_YEAR", weekofyear("dt"))
    .withColumn("MONTH_NUM", month("dt"))
    .withColumn("MONTH_NAME", date_format("dt", "MMMM"))
    .withColumn("QUARTER", quarter("dt"))
    .withColumn("YEAR", year("dt"))
    .withColumn("DI_CREATEDBY", lit("pipeline"))
    .drop("dt")
    .select("DATE_SK", "DI_CREATEDATE", "FULL_DT", "DAY_NUM", "DAY_NAME", "IS_WEEKEND",
            "WEEK_OF_YEAR", "MONTH_NUM", "MONTH_NAME", "QUARTER", "YEAR", "DI_CREATEDBY"))

dim_date.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.gold.dim_date")
print(f"dim_date: {dim_date.count()} rows"); display(dim_date.limit(5))

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

default_date_schema = StructType([
    StructField("DATE_SK", IntegerType()),
    StructField("DI_CREATEDATE", TimestampType()),
    StructField("FULL_DT", DateType()),
    StructField("DAY_NUM", IntegerType()),
    StructField("DAY_NAME", StringType()),
    StructField("IS_WEEKEND", StringType()),
    StructField("WEEK_OF_YEAR", IntegerType()),
    StructField("MONTH_NUM", IntegerType()),
    StructField("MONTH_NAME", StringType()),
    StructField("QUARTER", IntegerType()),
    StructField("YEAR", IntegerType()),
    StructField("DI_CREATEDBY", StringType()),
])

# Insert default "Unknown" row
from datetime import datetime
default_date = spark.createDataFrame(
    [(-9999, datetime.now(), None, -9999, "Unknown", "Unknown", -9999, -9999, "Unknown", -9999, -9999, "pipeline")],
    schema=default_date_schema
)
default_date.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_date")
print("  + Added default row (DATE_SK = -9999)")

### 4.2 — dim_inspection_type

In [0]:
chi_t = chi_insp.select(col("Inspection_Type"), lit("Chicago").alias("CITY")).distinct()
dal_t = dal_insp.select(col("Inspection_Type"), lit("Dallas").alias("CITY")).distinct()
all_t = chi_t.union(dal_t).distinct()

dim_type = (all_t
    .withColumn("INSPECTION_TYPE_SK", row_number().over(Window.orderBy("CITY","Inspection_Type")))
    .withColumn("INSPECTION_TYPE", col("Inspection_Type"))
    .withColumn("DI_CREATEDBY", lit("pipeline")).withColumn("DI_CREATEDATE", current_timestamp())
    .select("INSPECTION_TYPE_SK", "INSPECTION_TYPE", "CITY", "DI_CREATEDBY", "DI_CREATEDATE"))

dim_type.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.gold.dim_inspection_type")
print(f"dim_inspection_type: {dim_type.count()} rows"); display(dim_type.limit(10))

# Insert default "Unknown" row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

default_type = spark.createDataFrame(
    [(-9999, "Unknown", "Unknown", "pipeline", datetime.now())],
    schema=StructType([
        StructField("INSPECTION_TYPE_SK", IntegerType()),
        StructField("INSPECTION_TYPE", StringType()),
        StructField("CITY", StringType()),
        StructField("DI_CREATEDBY", StringType()),
        StructField("DI_CREATEDATE", TimestampType()),
    ])
)
default_type.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_inspection_type")
print("  + Added default row (INSPECTION_TYPE_SK = -9999)")

### 4.3 — dim_inspection_result

In [0]:
chi_r = chi_insp.select(col("Results").alias("RESULT_TYPE"), lit("Chicago").alias("CITY")).distinct()
dal_r = (dal_insp.select("Inspection_Score").distinct()
    .withColumn("RESULT_TYPE",
        when(col("Inspection_Score")>=90,"Pass").when(col("Inspection_Score")>=80,"Pass w/ Conditions")
        .when(col("Inspection_Score")>=70,"Marginal").otherwise("Fail"))
    .select("RESULT_TYPE").distinct().withColumn("CITY", lit("Dallas")))

all_r = chi_r.union(dal_r).distinct()
dim_result = (all_r
    .withColumn("INSPECTION_RESULT_SK", row_number().over(Window.orderBy("CITY","RESULT_TYPE")))
    .withColumn("DI_CREATEDBY", lit("pipeline")).withColumn("DI_CREATEDATE", current_timestamp())
    .select("INSPECTION_RESULT_SK", "CITY", "DI_CREATEDBY", "DI_CREATEDATE", "RESULT_TYPE"))

dim_result.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.gold.dim_inspection_result")
print(f"dim_inspection_result: {dim_result.count()} rows"); display(dim_result)

# Insert default "Unknown" row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

default_result = spark.createDataFrame(
    [(-9999, "Unknown", "pipeline", datetime.now(), "Unknown")],
    schema=StructType([
        StructField("INSPECTION_RESULT_SK", IntegerType()),
        StructField("CITY", StringType()),
        StructField("DI_CREATEDBY", StringType()),
        StructField("DI_CREATEDATE", TimestampType()),
        StructField("RESULT_TYPE", StringType()),
    ])
)
default_result.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_inspection_result")
print("  + Added default row (INSPECTION_RESULT_SK = -9999)")

### 4.4 — dim_location

In [0]:
chi_l = chi_insp.select(col("Address").alias("ADDRESS"), col("City").alias("CITY"),
    col("State").alias("STATE"), col("Zip").cast("string").alias("ZIP_CODE"),
    col("Latitude").alias("LATITUDE"), col("Longitude").alias("LONGITUDE")).distinct()

dal_l = dal_insp.select(col("Street_Address").alias("ADDRESS"), lit("Dallas").alias("CITY"),
    lit("TX").alias("STATE"), col("Zip_Code").cast("string").alias("ZIP_CODE"),
    col("Latitude").alias("LATITUDE"), col("Longitude").alias("LONGITUDE")).distinct()

all_l = chi_l.union(dal_l).distinct()
dim_loc = (all_l
    .withColumn("LOCATION_SK", row_number().over(Window.orderBy("CITY","ADDRESS","ZIP_CODE")))
    .withColumn("DI_CREATEDBY", lit("pipeline")).withColumn("DI_CREATEDATE", current_timestamp())
    .select("LOCATION_SK", "ADDRESS", "CITY", "STATE", "ZIP_CODE", "LATITUDE", "LONGITUDE", "DI_CREATEDBY", "DI_CREATEDATE"))

dim_loc = (dim_loc
    .withColumn("CITY", coalesce(col("CITY"), lit("Unknown")))
    .withColumn("STATE", coalesce(col("STATE"), lit("Unknown")))
    .withColumn("LATITUDE", coalesce(col("LATITUDE"), lit(-9999.0)))
    .withColumn("LONGITUDE", coalesce(col("LONGITUDE"), lit(-9999.0)))
)


dim_loc.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.gold.dim_location")
print(f"dim_location: {dim_loc.count()} rows")

# Insert default "Unknown" row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

default_loc = spark.createDataFrame(
    [(-9999, "Unknown", "Unknown", "Unknown", "Unknown", -9999.0, -9999.0, "pipeline", datetime.now())],
    schema=StructType([
        StructField("LOCATION_SK", IntegerType()),
        StructField("ADDRESS", StringType()),
        StructField("CITY", StringType()),
        StructField("STATE", StringType()),
        StructField("ZIP_CODE", StringType()),
        StructField("LATITUDE", DoubleType()),
        StructField("LONGITUDE", DoubleType()),
        StructField("DI_CREATEDBY", StringType()),
        StructField("DI_CREATEDATE", TimestampType()),
    ])
)
default_loc.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_location")
print("  + Added default row (LOCATION_SK = -9999)")

### 4.5 — dim_violation
Stores both Chicago and Dallas violation codes (they don't need to match).

In [0]:
chi_v = chi_viol.select(col("Violation_Code").alias("VIOLATION_CODE"),
    col("Violation_Description").alias("VIOLATION_DESC"),
    lit(None).cast("string").alias("VIOLATION_SEVERITY"), lit("Chicago").alias("CITY")).distinct()

dal_v = dal_viol.select(col("Violation_Code").alias("VIOLATION_CODE"),
    col("Violation_Description").alias("VIOLATION_DESC"),
    lit(None).cast("string").alias("VIOLATION_SEVERITY"), lit("Dallas").alias("CITY")).distinct()

all_v = chi_v.union(dal_v).distinct()
dim_viol = (all_v
    .withColumn("VIOLATION_SK", row_number().over(Window.orderBy("CITY","VIOLATION_CODE","VIOLATION_DESC")))
    .withColumn("DI_CREATEDBY", lit("pipeline")).withColumn("DI_CREATEDATE", current_timestamp())
    .select("VIOLATION_SK", "VIOLATION_CODE", "VIOLATION_DESC", "VIOLATION_SEVERITY", "CITY", "DI_CREATEDBY", "DI_CREATEDATE"))

dim_viol = (dim_viol
    .withColumn("VIOLATION_DESC", trim(regexp_replace(col("VIOLATION_DESC"), r"^\*?\s*\d+\s*", "")))
    .withColumn("VIOLATION_DESC", trim(regexp_replace(col("VIOLATION_DESC"), r"^\*\s*", "")))
    .withColumn("VIOLATION_CODE", coalesce(col("VIOLATION_CODE"), lit(-9999)))
    .withColumn("VIOLATION_DESC", coalesce(col("VIOLATION_DESC"), lit("Unknown")))
    .withColumn("VIOLATION_SEVERITY", coalesce(col("VIOLATION_SEVERITY"), lit("Unknown")))
)



dim_viol.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.gold.dim_violation")
print(f"dim_violation: {dim_viol.count()} rows"); display(dim_viol.limit(10))

# Insert default "Unknown" row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

default_viol = spark.createDataFrame(
    [(-9999, -9999, "Unknown", "Unknown", "Unknown", "pipeline", datetime.now())],
    schema=StructType([
        StructField("VIOLATION_SK", IntegerType()),
        StructField("VIOLATION_CODE", IntegerType()),
        StructField("VIOLATION_DESC", StringType()),
        StructField("VIOLATION_SEVERITY", StringType()),
        StructField("CITY", StringType()),
        StructField("DI_CREATEDBY", StringType()),
        StructField("DI_CREATEDATE", TimestampType()),
    ])
)
default_viol.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_violation")
print("  + Added default row (VIOLATION_SK = -9999)")

### 4.6 — dim_establishment (SCD Type 2)
Tracks historical changes using hash-based change detection.
SCD2 columns: `EFF_START_DATE`, `EFF_END_DATE`, `IS_CURRENT`

In [0]:
# Build incoming establishments from both cities
chi_est = chi_insp.select(
    col("License_Num").cast("string").alias("ESTABLISHMENT_NK"),
    col("DBA_Name").alias("DBA_NAME"),
    col("AKA_Name").alias("AKA_NAME"),
    col("License_Num").cast("string").alias("LICENSE_NUMBER"),
    col("Facility_Type").alias("FACILITY_TYPE"),
    col("Risk").alias("RISK_CATEGORY")).distinct()

dal_est = dal_insp.select(
    concat_ws("_", col("Restaurant_Name"), col("Zip_Code")).alias("ESTABLISHMENT_NK"),
    col("Restaurant_Name").alias("DBA_NAME"),
    lit(None).cast("string").alias("AKA_NAME"),
    lit(None).cast("string").alias("LICENSE_NUMBER"),
    lit(None).cast("string").alias("FACILITY_TYPE"),
    lit(None).cast("string").alias("RISK_CATEGORY")).distinct()

df_incoming = chi_est.union(dal_est).distinct()
df_incoming = df_incoming.withColumn("row_hash", md5(concat_ws("||",
    coalesce(col("DBA_NAME"), lit("")),
    coalesce(col("AKA_NAME"), lit("")),
    coalesce(col("LICENSE_NUMBER"), lit("")),
    coalesce(col("FACILITY_TYPE"), lit("")),
    coalesce(col("RISK_CATEGORY"), lit("")))))

table_exists = spark.catalog.tableExists(f"{CATALOG}.gold.dim_establishment")

if not table_exists:
    # ── INITIAL LOAD: all records are new ──
    dim_est = (df_incoming
        .withColumn("ESTABLISHMENT_SK", row_number().over(Window.orderBy("ESTABLISHMENT_NK")))
        .withColumn("EFF_START_DATE", current_date())
        .withColumn("EFF_END_DATE", lit(None).cast("date"))
        .withColumn("IS_CURRENT", lit("Y"))
        .withColumn("DI_CREATEDBY", lit("pipeline"))
        .withColumn("DI_CREATEDATE", current_timestamp())
        .select("ESTABLISHMENT_SK", "ESTABLISHMENT_NK", "DBA_NAME", "AKA_NAME", "LICENSE_NUMBER",
                "FACILITY_TYPE", "RISK_CATEGORY", "EFF_START_DATE", "EFF_END_DATE", "IS_CURRENT",
                "DI_CREATEDBY", "DI_CREATEDATE", "row_hash"))
    
    # Apply Kimball sentinel values BEFORE saving
    dim_est = (dim_est
        .withColumn("AKA_NAME", coalesce(col("AKA_NAME"), lit("Unknown")))
        .withColumn("LICENSE_NUMBER", coalesce(col("LICENSE_NUMBER"), lit("Unknown")))
        .withColumn("FACILITY_TYPE", coalesce(col("FACILITY_TYPE"), lit("Unknown")))
        .withColumn("RISK_CATEGORY", coalesce(col("RISK_CATEGORY"), lit("Unknown")))
        .withColumn("ESTABLISHMENT_NK", coalesce(col("ESTABLISHMENT_NK"), lit("Unknown")))
    )


    dim_est.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_establishment")
    print(f"dim_establishment (initial load): {dim_est.count()} rows")

    

else:
    # ── SCD2 MERGE: detect new + changed records ──
    existing = spark.table(f"{CATALOG}.gold.dim_establishment").filter("IS_CURRENT = 'Y'")
    max_sk = existing.agg(F.max("ESTABLISHMENT_SK")).collect()[0][0] or 0

    # Find NEW records (not in existing) and CHANGED records (hash mismatch)
    changes = (df_incoming.alias("n")
        .join(existing.alias("o"), col("n.ESTABLISHMENT_NK") == col("o.ESTABLISHMENT_NK"), "left")
        .filter(
            col("o.ESTABLISHMENT_NK").isNull() |       # new establishment
            (col("n.row_hash") != col("o.row_hash"))   # attributes changed
        )
        .select("n.*"))

    cnt = changes.count()
    print(f"Changes detected: {cnt}")

    if cnt > 0:
        # Step 1: CLOSE old records — set IS_CURRENT='N' and EFF_END_DATE=today
        changes.createOrReplaceTempView("changes_temp")
        spark.sql(f"""
            UPDATE {CATALOG}.gold.dim_establishment
            SET IS_CURRENT = 'N', EFF_END_DATE = current_date()
            WHERE IS_CURRENT = 'Y'
            AND ESTABLISHMENT_NK IN (SELECT ESTABLISHMENT_NK FROM changes_temp)
        """)

        # Step 2: INSERT new current records with new SK
        new_recs = (changes
            .withColumn("ESTABLISHMENT_SK", row_number().over(Window.orderBy("ESTABLISHMENT_NK")) + lit(max_sk))
            .withColumn("EFF_START_DATE", current_date())
            .withColumn("EFF_END_DATE", lit(None).cast("date"))
            .withColumn("IS_CURRENT", lit("Y"))
            .withColumn("DI_CREATEDBY", lit("pipeline"))
            .withColumn("DI_CREATEDATE", current_timestamp())
            .select("ESTABLISHMENT_SK", "ESTABLISHMENT_NK", "DBA_NAME", "AKA_NAME", "LICENSE_NUMBER",
                    "FACILITY_TYPE", "RISK_CATEGORY", "EFF_START_DATE", "EFF_END_DATE", "IS_CURRENT",
                    "DI_CREATEDBY", "DI_CREATEDATE", "row_hash"))
        
         # Apply Kimball sentinel values BEFORE appending
        new_recs = (new_recs
            .withColumn("AKA_NAME", coalesce(col("AKA_NAME"), lit("Unknown")))
            .withColumn("LICENSE_NUMBER", coalesce(col("LICENSE_NUMBER"), lit("Unknown")))
            .withColumn("FACILITY_TYPE", coalesce(col("FACILITY_TYPE"), lit("Unknown")))
            .withColumn("RISK_CATEGORY", coalesce(col("RISK_CATEGORY"), lit("Unknown")))
            .withColumn("ESTABLISHMENT_NK", coalesce(col("ESTABLISHMENT_NK"), lit("Unknown")))
        )

        new_recs.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_establishment")
        print(f"  Closed old records and inserted {new_recs.count()} new SCD2 records")
        
    else:
        print("  No changes detected — dim_establishment unchanged")

# ── Summary ──
est = spark.table(f"{CATALOG}.gold.dim_establishment")
total = est.count()
current = est.filter("IS_CURRENT = 'Y'").count()
historical = est.filter("IS_CURRENT = 'N'").count()
print(f"dim_establishment — Total: {total}, Current: {current}, Historical: {historical}")

# ── Insert default "Unknown" row (only if not already present) ──
existing_default = spark.table(f"{CATALOG}.gold.dim_establishment").filter("ESTABLISHMENT_SK = -9999")
if existing_default.count() == 0:
    from datetime import datetime, date
    from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

    default_est = spark.createDataFrame(
        [(-9999, "Unknown", "Unknown", "Unknown", "Unknown", "Unknown", "Unknown",
          date.today(), None, "Y", "pipeline", datetime.now(), "default")],
        schema=StructType([
            StructField("ESTABLISHMENT_SK", IntegerType()),
            StructField("ESTABLISHMENT_NK", StringType()),
            StructField("DBA_NAME", StringType()),
            StructField("AKA_NAME", StringType()),
            StructField("LICENSE_NUMBER", StringType()),
            StructField("FACILITY_TYPE", StringType()),
            StructField("RISK_CATEGORY", StringType()),
            StructField("EFF_START_DATE", DateType()),
            StructField("EFF_END_DATE", DateType()),
            StructField("IS_CURRENT", StringType()),
            StructField("DI_CREATEDBY", StringType()),
            StructField("DI_CREATEDATE", TimestampType()),
            StructField("row_hash", StringType()),
        ])
    )
    default_est.write.mode("append").saveAsTable(f"{CATALOG}.gold.dim_establishment")
    print("  + Added default row (ESTABLISHMENT_SK = -9999)")

### 4.7 — fact_inspection

In [0]:
dim_est = spark.table(f"{CATALOG}.gold.dim_establishment").filter("IS_CURRENT='Y'")
dim_loc = spark.table(f"{CATALOG}.gold.dim_location")
dim_dt = spark.table(f"{CATALOG}.gold.dim_date")
dim_tp = spark.table(f"{CATALOG}.gold.dim_inspection_type")
dim_rs = spark.table(f"{CATALOG}.gold.dim_inspection_result")

# Chicago
chi_f = (chi_insp
    .withColumn("INSPECTION_ID", col("Inspection_ID"))
    .withColumn("INSPECTION_SCORE", col("Violation_Score"))
    .withColumn("_nk", col("License_Num").cast("string"))
    .withColumn("_city", lit("Chicago"))
    .withColumn("_result", col("Results"))
    .select("INSPECTION_ID", "INSPECTION_SCORE", "_nk", "_city", "_result",
            col("Inspection_Date").alias("_dt"), col("Inspection_Type").alias("_type"),
            col("Address").alias("_addr"), col("Zip").cast("string").alias("_zip")))

# Dallas
dal_f = (dal_insp
    .withColumn("INSPECTION_ID", monotonically_increasing_id() + 10000000)
    .withColumn("INSPECTION_SCORE", col("Inspection_Score"))
    .withColumn("_nk", concat_ws("_", col("Restaurant_Name"), col("Zip_Code")))
    .withColumn("_city", lit("Dallas"))
    .withColumn("_result",
        when(col("Inspection_Score") >= 90, "Pass")
        .when(col("Inspection_Score") >= 80, "Pass w/ Conditions")
        .when(col("Inspection_Score") >= 70, "Marginal")
        .otherwise("Fail"))
    .select("INSPECTION_ID", "INSPECTION_SCORE", "_nk", "_city", "_result",
            col("Inspection_Date").alias("_dt"), col("Inspection_Type").alias("_type"),
            col("Street_Address").alias("_addr"), col("Zip_Code").alias("_zip")))

all_f = chi_f.union(dal_f)

# Join to all dimensions for SK lookups
fact = (all_f
    .join(dim_dt.select("DATE_SK", "FULL_DT"), all_f["_dt"] == dim_dt["FULL_DT"], "left")
    .join(dim_est.select("ESTABLISHMENT_SK", "ESTABLISHMENT_NK"), all_f["_nk"] == dim_est["ESTABLISHMENT_NK"], "left")
    .join(dim_loc.select("LOCATION_SK", "ADDRESS", "ZIP_CODE"),
          (all_f["_addr"] == dim_loc["ADDRESS"]) & (all_f["_zip"] == dim_loc["ZIP_CODE"]), "left")
    .join(dim_tp.select("INSPECTION_TYPE_SK", col("INSPECTION_TYPE").alias("_dtp"), col("CITY").alias("_dtpc")),
          (all_f["_type"] == col("_dtp")) & (all_f["_city"] == col("_dtpc")), "left")
    .join(dim_rs.select("INSPECTION_RESULT_SK", col("RESULT_TYPE").alias("_drs"), col("CITY").alias("_drsc")),
          (all_f["_result"] == col("_drs")) & (all_f["_city"] == col("_drsc")), "left")
    .withColumn("INSPECTION_SK", monotonically_increasing_id())
    .withColumn("ESTABLISHMENT_SK", coalesce(col("ESTABLISHMENT_SK"), lit(-9999)))
    .withColumn("LOCATION_SK", coalesce(col("LOCATION_SK"), lit(-9999)))
    .withColumn("DATE_SK", coalesce(col("DATE_SK"), lit(-9999)))
    .withColumn("INSPECTION_TYPE_SK", coalesce(col("INSPECTION_TYPE_SK"), lit(-9999)))
    .withColumn("INSPECTION_RESULT_SK", coalesce(col("INSPECTION_RESULT_SK"), lit(-9999)))
    .withColumn("INSPECTION_SCORE", coalesce(col("INSPECTION_SCORE"), lit(-9999)))
    .withColumn("DI_CREATEDBY", lit("pipeline"))
    .withColumn("DI_CREATEDATE", current_timestamp())
    .select("INSPECTION_SK", "LOCATION_SK", "ESTABLISHMENT_SK", "DATE_SK",
            "INSPECTION_RESULT_SK", "INSPECTION_TYPE_SK",
            "DI_CREATEDATE", "INSPECTION_ID", "INSPECTION_SCORE", "DI_CREATEDBY"))

# Check if fact table exists
fact_table_exists = spark.catalog.tableExists(f"{CATALOG}.gold.fact_inspection")

if not fact_table_exists:
    # First load — create the table
    fact.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.fact_inspection")
    print(f"fact_inspection (initial load): {fact.count()} rows")
else:
    # Subsequent loads — append only NEW inspections (avoid duplicates)
    existing_fact = spark.table(f"{CATALOG}.gold.fact_inspection")
    
    # Find inspections not already in the fact table
    new_facts = fact.join(
        existing_fact.select("INSPECTION_ID"),
        "INSPECTION_ID",
        "left_anti"
    )
    
    new_count = new_facts.count()
    if new_count > 0:
        new_facts.write.mode("append").saveAsTable(f"{CATALOG}.gold.fact_inspection")
        print(f"fact_inspection — Appended {new_count} new rows")
    else:
        print("fact_inspection — No new inspections to load")

total = spark.table(f"{CATALOG}.gold.fact_inspection").count()
print(f"fact_inspection — Total: {total} rows")
display(spark.table(f"{CATALOG}.gold.fact_inspection").limit(10))

### 4.8 — bridge_inspection_violation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

CATALOG = "chicago_dallas_food_inspection"

chi_insp = spark.table(f"{CATALOG}.silver.chicago_inspections")
dal_insp = spark.table(f"{CATALOG}.silver.dallas_inspections")
chi_viol = spark.table(f"{CATALOG}.silver.chicago_violations")
dal_viol = spark.table(f"{CATALOG}.silver.dallas_violations")

In [0]:
fact_df = spark.table(f"{CATALOG}.gold.fact_inspection")
dim_v = spark.table(f"{CATALOG}.gold.dim_violation")

# Deduplicate dim_violation to one SK per VIOLATION_CODE + CITY
dim_v_dedup = (dim_v
    .withColumn("_rn", row_number().over(Window.partitionBy("VIOLATION_CODE", "CITY").orderBy("VIOLATION_SK")))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select("VIOLATION_SK", col("VIOLATION_CODE").alias("_dvc"), col("CITY").alias("_dvcity"))
)

# ── Chicago bridge ──
chi_bridge = (chi_viol
    .select(
        col("Inspection_ID").cast("long").alias("_iid"),
        col("Violation_Code").alias("_vc"),
        lit("Chicago").alias("_city"),
        col("Violation_Comments").alias("VIOLATION_MEMO")
    )
    .withColumn("VIOLATION_POINTS", lit(None).cast("int"))
    .join(fact_df.select("INSPECTION_SK", "INSPECTION_ID"), col("_iid") == col("INSPECTION_ID"), "left")
    .join(dim_v_dedup, (col("_vc") == col("_dvc")) & (col("_city") == col("_dvcity")), "left")
    .withColumn("VIOLATION_SK", coalesce(col("VIOLATION_SK"), lit(-9999)))
    .withColumn("INSPECTION_SK", coalesce(col("INSPECTION_SK"), lit(-9999)))
    .withColumn("VIOLATION_POINTS", coalesce(col("VIOLATION_POINTS"), lit(-9999)))
    .withColumn("VIOLATION_MEMO", coalesce(col("VIOLATION_MEMO"), lit("Unknown")))
    .withColumn("DI_CREATEDBY", lit("pipeline"))
    .withColumn("DI_CREATEDATE", current_timestamp())
    .select("VIOLATION_SK", "INSPECTION_SK", "VIOLATION_POINTS", "VIOLATION_MEMO", "DI_CREATEDBY", "DI_CREATEDATE")
)

# ── Dallas bridge — skip fact join, use -9999 for INSPECTION_SK ──
# Dallas has no reliable Inspection_ID to join to fact_inspection
# The violation details (code, points, memo) are still linked correctly via dim_violation
dal_bridge = (dal_viol
    .select(
        col("Violation_Code").alias("_vc"),
        lit("Dallas").alias("_city"),
        col("Violation_Comments").alias("VIOLATION_MEMO"),
        col("Violation_Points").alias("VIOLATION_POINTS")
    )
    .join(dim_v_dedup, (col("_vc") == col("_dvc")) & (col("_city") == col("_dvcity")), "left")
    .withColumn("VIOLATION_SK", coalesce(col("VIOLATION_SK"), lit(-9999)))
    .withColumn("INSPECTION_SK", lit(-9999))
    .withColumn("VIOLATION_POINTS", coalesce(col("VIOLATION_POINTS"), lit(-9999)))
    .withColumn("VIOLATION_MEMO", coalesce(col("VIOLATION_MEMO"), lit("Unknown")))
    .withColumn("DI_CREATEDBY", lit("pipeline"))
    .withColumn("DI_CREATEDATE", current_timestamp())
    .select("VIOLATION_SK", "INSPECTION_SK", "VIOLATION_POINTS", "VIOLATION_MEMO", "DI_CREATEDBY", "DI_CREATEDATE")
)

# Union both cities
bridge = chi_bridge.union(dal_bridge)

# Save
bridge.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.bridge_inspection_violation")

# Validate
total = spark.table(f"{CATALOG}.gold.bridge_inspection_violation").count()
sentinel_viol = spark.table(f"{CATALOG}.gold.bridge_inspection_violation").filter(col("VIOLATION_SK") == -9999).count()
sentinel_insp = spark.table(f"{CATALOG}.gold.bridge_inspection_violation").filter(col("INSPECTION_SK") == -9999).count()
print(f"bridge_inspection_violation — Total: {total}")
print(f"  VIOLATION_SK = -9999: {sentinel_viol}")
print(f"  INSPECTION_SK = -9999: {sentinel_insp}")
display(spark.table(f"{CATALOG}.gold.bridge_inspection_violation").limit(10))

### 4.9 — Gold Zone Summary

In [0]:
print("=" * 70); print("GOLD ZONE — DIMENSIONAL MODEL"); print("=" * 70)
for t in ["dim_date","dim_inspection_type","dim_inspection_result","dim_location",
          "dim_violation","dim_establishment","fact_inspection","bridge_inspection_violation"]:
    print(f"  {CATALOG}.gold.{t}: {spark.table(f'{CATALOG}.gold.{t}').count()} rows")

est = spark.table(f"{CATALOG}.gold.dim_establishment")
total = est.count()
current = est.filter("IS_CURRENT = 'Y'").count()
historical = est.filter("IS_CURRENT = 'N'").count()
print(f"\ndim_establishment SCD2: Total={total}, Current={current}, Historical={historical}")
print("=" * 70)